# 9. Bird vs Drone Image Classification

## 9.1 Import required dependencies and modules

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPool2D
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.models import load_model

In [ ]:
import random
import os
import shutil
import zipfile
import pathlib

## 9.2 Mount Google Drive

In [52]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


## 9.3 Download Dataset from Kaggle

In [7]:
! pip install kaggle

In [8]:
source = "/content/drive/MyDrive/kaggle.json"
destination = "/content/"

shutil.copy(src=source, dst=destination)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/kaggle.json'

In [9]:
! mkdir ~/.kaggle

In [10]:
! cp kaggle.json ~/.kaggle/

cp: cannot stat 'kaggle.json': No such file or directory


In [11]:
! chmod 600 ~/.kaggle/kaggle.json

chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [12]:
! kaggle datasets download harshwalia/birds-vs-drone-dataset

Traceback (most recent call last):
  File "/usr/local/bin/kaggle", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/kaggle/cli.py", line 68, in main
    out = args.func(**command_args)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/kaggle/api/kaggle_api_extended.py", line 1741, in dataset_download_cli
    with self.build_kaggle_client() as kaggle:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/kaggle/api/kaggle_api_extended.py", line 688, in build_kaggle_client
    username=self.config_values['username'],
             ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'username'


In [13]:
# Unzip the downloaded file
zip_ref = zipfile.ZipFile("birds-vs-drone-dataset.zip")
zip_ref.extractall()
zip_ref.close()

FileNotFoundError: [Errno 2] No such file or directory: 'birds-vs-drone-dataset.zip'

## 9.4 Split the dataset into training and test sets

In [14]:
def split_data(SOURCE, TRAINING, TESTING, SPLIT_SIZE):
    file_names = os.listdir(SOURCE)
    random_file_names = random.sample(file_names, len(file_names))
    split_threshold = round(SPLIT_SIZE * len(random_file_names))
    for item in random_file_names[:split_threshold]:
        if os.path.getsize(os.path.join(SOURCE,item)) == 0:
            print(f"{item} is zero length, so ignoring.")
        else:
            shutil.copy(os.path.join(SOURCE,item),TRAINING)
    for item in random_file_names[split_threshold:len(random_file_names)]:
        if os.path.getsize(os.path.join(SOURCE,item)) == 0:
            print(f"{item} is zero length, so ignoring.")
        else:
            shutil.copy(os.path.join(SOURCE,item),TESTING)

In [18]:
DRONE_SOURCE_DIR = "/content/BirdVsDrone/Drones"
BIRD_SOURCE_DIR = "/content/BirdVsDrone/Birds"

TRAINING_DIR = '/content/BirdVsDroneSplit/train/'
TESTING_DIR = '/content/BirdVsDroneSplit/test/'

TRAINING_DRONE_DIR = TRAINING_DIR + "drones/"
TESTING_DRONE_DIR = TESTING_DIR + "drones/"

TRAINING_BIRD_DIR = TRAINING_DIR + "birds/"
TESTING_BIRD_DIR = TESTING_DIR + "birds/"

In [16]:
try:
  os.makedirs(TRAINING_BIRD_DIR)
  os.makedirs(TESTING_BIRD_DIR)
  os.makedirs(TRAINING_DRONE_DIR)
  os.makedirs(TESTING_DRONE_DIR)
except:
  pass

In [17]:
split_size = 0.8

split_data(DRONE_SOURCE_DIR, TRAINING_DRONE_DIR, TESTING_DRONE_DIR, split_size)
split_data(BIRD_SOURCE_DIR, TRAINING_BIRD_DIR, TESTING_BIRD_DIR, split_size)

FileNotFoundError: [Errno 2] No such file or directory: '/content/BirdVsDrone/Drones'

## 9.5 Load the split dataset from Google Drive

In [19]:
dataset_path = "/content/drive/MyDrive/BirdVsDroneSplit"

for dirpath, dirnames, filenames in os.walk(dataset_path):
    print(f"There are {len(dirnames)} directories and {len(filenames)} images in '{dirpath}'.")

In [20]:
# Get the class names programmatically
data_dir = pathlib.Path(dataset_path + "/train")
class_names = np.array(sorted([item.name for item in data_dir.glob("*")])) # Created a list of class_names from the subdirectories
print(class_names)

[]


## 9.6 Visualize random images from dataset

In [21]:
# Let's visualize our images
import matplotlib.image as mpimg

def view_random_image(target_dir, target_class):
    # Setup the target directory (we'll view images from here)
    target_folder = target_dir + target_class

    # Get a random image path
    random_image = random.sample(os.listdir(target_folder), 1)
    print(random_image)

    # Read in the image and plot it using matplotlib
    img = mpimg.imread(target_folder + "/" + random_image[0])
    plt.imshow(img)
    plt.title(target_class)
    plt.axis("off");

    print(f"Image shape: {img.shape}") # show the shape of the image

    return img

In [22]:
# View a random image from the training dataset
img = view_random_image(target_dir=dataset_path + "/train/",
                        target_class="birds")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/BirdVsDroneSplit/train/birds'

In [23]:
# View a random image from the training dataset
img = view_random_image(target_dir=dataset_path + "/train/",
                        target_class="drones")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/BirdVsDroneSplit/train/drones'

## 9.7 Prepare the dataset to feed into the neural network

In [24]:
train_datagen = ImageDataGenerator(rescale=1/255.0)
test_datagen = ImageDataGenerator(rescale=1/255.0)

In [25]:
train_dir = dataset_path + "/train/"
test_dir = dataset_path + "/test/"

In [26]:
train_data = train_datagen.flow_from_directory(directory=train_dir,     # Target directory of images
                                               target_size=(224, 224),  # Target size of images (height, width)
                                               class_mode="binary",     # Type of data you're working with
                                               batch_size=32,           # Size of minibatches to load data into
                                               shuffle=True)

test_data = test_datagen.flow_from_directory(directory=test_dir,
                                             target_size=(224, 224),
                                             class_mode="binary",
                                             batch_size=32,
                                             shuffle=True)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/BirdVsDroneSplit/train/'

In [27]:
train_datagen_augmented = ImageDataGenerator(rescale=1/255.,
                                             rotation_range=0.2,        # how much do you want to rotate an image?
                                             shear_range=0.2,           # how much do you want to shear an image?
                                             zoom_range=0.2,            # zoom in randomly on an image
                                             width_shift_range=0.2,     # move your image around on the x-axis
                                             height_shift_range=0.2,    # move your image around on the y-axis
                                             horizontal_flip=True)      # do you want to flip an image?

In [28]:
train_data_augmented = train_datagen_augmented.flow_from_directory(directory=train_dir,
                                                                   target_size=(224, 224),
                                                                   class_mode="binary",
                                                                   batch_size=32)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/BirdVsDroneSplit/train/'

In [29]:
saved_models_path = "/content/drive/MyDrive/saved_models/"

## 9.8 Define helper functions

In [30]:
# Plot the validation and training curves separately
def plot_loss_curves(history):
    """
    Returns separate loss curves for training and validation metrics
    """
    loss = history.history["loss"]
    val_loss = history.history["val_loss"]

    accuracy = history.history["accuracy"]
    val_accuracy = history.history["val_accuracy"]

    epochs = range(len(history.history["loss"]))    # how many epochs did we run for?

    # Plot loss
    plt.plot(epochs, loss, label="training_loss")
    plt.plot(epochs, val_loss, label="val_loss")
    plt.title("Loss")
    plt.xlabel("Epochs")
    plt.legend()

    # Plot accuracy
    plt.figure()
    plt.plot(epochs, accuracy, label="training_accuracy")
    plt.plot(epochs, val_accuracy, label="val_accuracy")
    plt.title("Accuracy")
    plt.xlabel("Epochs")
    plt.legend()

In [31]:
# Create a function to import an image and resize it to be able to be used with our model
def load_and_prep_image(filename, img_shape=224):
    """
    Reads an image from filename, turns it into a tensor and reshapes it
    to (img_shape, img_shape, colour_channels).
    """
    # Read in the image
    img = tf.io.read_file(filename)

    # Decode the read file into a tensor
    img = tf.image.decode_image(img)

    # Resize the image
    img = tf.image.resize(img, size=[img_shape, img_shape])

    # Rescale the image (get all values between 0 and 1)
    img = img / 255.

    return img

In [32]:
def pred_and_plot(model, filename, class_names=class_names):
    """
    Imports an image located at filename, makes a prediction with model
    and plots the image with the predicted class as the title.
    """
    # Import the target image and preprocess it
    img = load_and_prep_image(filename)

    # Make a prediction
    pred = model.predict(tf.expand_dims(img, axis=0))
    print(pred)

    # Get the predicted class
    pred_class = class_names[int(tf.round(pred))]

    # Plot the image and predicted class
    plt.imshow(img)
    plt.title(f"Prediction: {pred_class}")
    plt.axis(False);

In [33]:
def pred_and_plot_directory(model, directory, class_names=class_names):
    """
    Imports an image located at directory, makes a prediction with model
    and plots the image with the predicted class as the title.
    """
    # Import the target image and preprocess it
    plt.figure(figsize=(12, 12))
    i = 1
    for filename in os.listdir(directory):
        img = load_and_prep_image(directory + "/" + filename)

        # Make a prediction
        pred = model.predict(tf.expand_dims(img, axis=0), verbose=0)
        prob = pred[0][0]
        # print(pred)

        # Get the predicted class
        pred_class = class_names[int(tf.round(pred))]

        # Calculate probability if pred_class is bird
        if pred_class == 'birds':
            prob = 1 - prob

        # Plot the image and predicted class
        plt.subplot(2, 2, i)
        plt.imshow(img)
        plt.title("Prediction: {}\nProbability: {:.4f}".format(pred_class, prob))
        plt.axis(False);
        i += 1

In [35]:
def walk_through_dir(dir_path):
  """
  Walks through dir_path returning its contents.
  Args:
    dir_path (str): target directory

  Returns:
    A print out of:
      number of subdiretories in dir_path
      number of images (files) in each subdirectory
      name of each subdirectory
  """
  for dirpath, dirnames, filenames in os.walk(dir_path):
    print(f"There are {len(dirnames)} directories and {len(filenames)} images in '{dirpath}'.")

## 9.9 Create the model, Compile the model and Fit the model

In [36]:
tf.random.set_seed(42)

model_1 = Sequential([
    # Note the input shape is the desired size of the image 224 * 224 with 3 bytes color
    # This is the first convolution
    Conv2D(16, (3,3), activation='relu', input_shape=(224, 224, 3)),
    MaxPool2D(2, 2),
    # The second convolution
    Conv2D(32, (3,3), activation='relu'),
    MaxPool2D(2,2),
    # The third convolution
    Conv2D(64, (3,3), activation='relu'),
    MaxPool2D(2,2),
    # Flatten the results to feed into a DNN
    Flatten(),
    # 512 neuron hidden layer
    Dense(512, activation='relu'),
    # Only 1 output neuron. It will contain a value from 0-1 where 0 for 1 class ('drones') and 1 for the other ('birds')
    Dense(1, activation='sigmoid')
])

model_1.compile(loss="binary_crossentropy",
                optimizer=Adam(),
                metrics=["accuracy"])

callback = ModelCheckpoint(filepath=saved_models_path + "/model_1",
                           monitor="val_accuracy",
                           save_best_only=True,
                           mode="max")

history_1 = model_1.fit(train_data_augmented,
                    epochs=110,
                    steps_per_epoch=len(train_data_augmented),
                    validation_data=test_data,
                    validation_steps=len(test_data),
                    callbacks=[callback])

plot_loss_curves(history_1)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


ValueError: The filepath provided must end in `.keras` (Keras model format). Received: filepath=/content/drive/MyDrive/saved_models//model_1

In [37]:
tf.random.set_seed(42)

model_2 = Sequential([
    # Note the input shape is the desired size of the image 224 * 224 with 3 bytes color
    # This is the first convolution
    Conv2D(16, (3,3), activation='relu', input_shape=(224, 224, 3)),
    Conv2D(16, (3,3), activation='relu'),
    MaxPool2D(2, 2),
    # The second convolution
    Conv2D(32, (3,3), activation='relu'),
    Conv2D(32, (3,3), activation='relu'),
    MaxPool2D(2,2),
    # The third convolution
    Conv2D(64, (3,3), activation='relu'),
    Conv2D(64, (3,3), activation='relu'),
    MaxPool2D(2,2),
    # Flatten the results to feed into a DNN
    Flatten(),
    # 512 neuron hidden layer
    Dense(256, activation='relu'),
    Dense(512, activation='relu'),
    # Only 1 output neuron. It will contain a value from 0-1 where 0 for 1 class ('drones') and 1 for the other ('birds')
    Dense(1, activation='sigmoid')
])

model_2.compile(loss="binary_crossentropy",
                optimizer=Adam(),
                metrics=["accuracy"])

callback = ModelCheckpoint(filepath=saved_models_path + "/model_2",
                           monitor="val_accuracy",
                           save_best_only=True,
                           mode="max")

history_2 = model_2.fit(train_data_augmented,
                    epochs=120,
                    steps_per_epoch=len(train_data_augmented),
                    validation_data=test_data,
                    validation_steps=len(test_data),
                    callbacks=[callback])

plot_loss_curves(history_2)

ValueError: The filepath provided must end in `.keras` (Keras model format). Received: filepath=/content/drive/MyDrive/saved_models//model_2

In [38]:
tf.random.set_seed(42)

model_3 = Sequential([
    # Note the input shape is the desired size of the image 224 * 224 with 3 bytes color
    # This is the first convolution
    Conv2D(16, (3,3), activation='relu', input_shape=(224, 224, 3)),
    Conv2D(16, (3,3), activation='relu'),
    MaxPool2D(2, 2),
    # The second convolution
    Conv2D(32, (4,4), activation='relu'),
    Conv2D(32, (4,4), activation='relu'),
    MaxPool2D(2,2),
    # The third convolution
    Conv2D(64, (5,5), activation='relu'),
    Conv2D(64, (5,5), activation='relu'),
    MaxPool2D(2,2),
    # Flatten the results to feed into a DNN
    Flatten(),
    # 512 neuron hidden layer
    Dense(256, activation='relu'),
    Dense(512, activation='relu'),
    # Only 1 output neuron. It will contain a value from 0-1 where 0 for 1 class ('drones') and 1 for the other ('birds')
    Dense(1, activation='sigmoid')
])

model_3.compile(loss="binary_crossentropy",
                optimizer=Adam(),
                metrics=["accuracy"])

callback = ModelCheckpoint(filepath=saved_models_path + "/model_3",
                           monitor="val_accuracy",
                           save_best_only=True,
                           mode="max")

history_3 = model_3.fit(train_data_augmented,
                    epochs=120,
                    steps_per_epoch=len(train_data_augmented),
                    validation_data=test_data,
                    validation_steps=len(test_data),
                    callbacks=[callback])

plot_loss_curves(history_3)

ValueError: The filepath provided must end in `.keras` (Keras model format). Received: filepath=/content/drive/MyDrive/saved_models//model_3

In [39]:
tf.random.set_seed(42)

model_4 = Sequential([
    # Note the input shape is the desired size of the image 224 * 224 with 3 bytes color
    # This is the first convolution
    Conv2D(16, (3,3), activation='relu', input_shape=(224, 224, 3)),
    Conv2D(16, (3,3), activation='relu'),
    MaxPool2D(2,2),
    # The second convolution
    Conv2D(32, (4,4), activation='relu'),
    Conv2D(32, (4,4), activation='relu'),
    MaxPool2D(2,2),
    # The third convolution
    Conv2D(64, (5,5), activation='relu'),
    Conv2D(64, (5,5), activation='relu'),
    MaxPool2D(2,2),
    # Flatten the results to feed into a DNN
    Flatten(),
    # 512 neuron hidden layer
    Dense(256, activation='relu'),
    Dense(512, activation='relu'),
    # Only 1 output neuron. It will contain a value from 0-1 where 0 for 1 class ('drones') and 1 for the other ('birds')
    Dense(1, activation='sigmoid')
])

model_4.compile(loss="binary_crossentropy",
                optimizer=Adam(),
                metrics=["accuracy"])

callback = ModelCheckpoint(filepath=saved_models_path + "/model_4",
                           monitor="val_accuracy",
                           save_best_only=True,
                           mode="max")

history_4 = model_4.fit(train_data_augmented,
                    epochs=150,
                    steps_per_epoch=len(train_data_augmented),
                    validation_data=test_data,
                    validation_steps=len(test_data),
                    callbacks=[callback])

plot_loss_curves(history_4)

ValueError: The filepath provided must end in `.keras` (Keras model format). Received: filepath=/content/drive/MyDrive/saved_models//model_4

## 9.10 Load the saved model

In [40]:
saved_model = load_model(saved_models_path + "/model_2")

ValueError: File format not supported: filepath=/content/drive/MyDrive/saved_models//model_2. Keras 3 only supports V3 `.keras` files and legacy H5 format files (`.h5` extension). Note that the legacy SavedModel format is not supported by `load_model()` in Keras 3. In order to reload a TensorFlow SavedModel as an inference-only layer in Keras 3, use `keras.layers.TFSMLayer(/content/drive/MyDrive/saved_models//model_2, call_endpoint='serving_default')` (note that your `call_endpoint` might have a different name).

In [41]:
saved_model.summary()

NameError: name 'saved_model' is not defined

In [42]:
from tensorflow.keras.utils import plot_model
plot_model(saved_model, show_shapes=True)

NameError: name 'saved_model' is not defined

## 9.11 Evaluate on training data, augmented training data and test data

In [43]:
saved_model.evaluate(train_data)

NameError: name 'saved_model' is not defined

In [44]:
saved_model.evaluate(train_data_augmented)

NameError: name 'saved_model' is not defined

In [45]:
saved_model.evaluate(test_data)

NameError: name 'saved_model' is not defined

## 9.12 Evaluate on custom images over the web


In [46]:
custom_birds = "/content/drive/MyDrive/custom_test/birds"
custom_drones = "/content/drive/MyDrive/custom_test/drones"

birds_images = os.listdir(custom_birds)
drones_images = os.listdir(custom_drones)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/custom_test/birds'

In [51]:
walk_through_dir(custom_birds)

In [48]:
pred_and_plot_directory(saved_model, custom_birds)

NameError: name 'saved_model' is not defined

In [49]:
walk_through_dir(custom_drones)

In [50]:
pred_and_plot_directory(saved_model, custom_drones)

NameError: name 'saved_model' is not defined